# Phase 2: 듀얼 벡터 스토어

`DualUpstashStore` 하나로 capability / experience 인제스트 + 검색 + 가중 합산까지 테스트한다.

In [2]:
from _bootstrap import setup_project_path

setup_project_path()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [4]:
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_embeddings
from embedding_retrieval.stores.dual_upstash import DualUpstashStore
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES

config = RetrievalConfig.from_env()
embeddings = create_embeddings(config)
url = config.vector_store_kwargs["url"]
token = config.vector_store_kwargs["token"]

# 듀얼 스토어 생성 — 내부에서 capability / experience 네임스페이스 자동 관리
store = DualUpstashStore(embeddings=embeddings, url=url, token=token)

# 프로필 인제스트 (한 번에)
store.add_profiles(SAMPLE_ENGINEER_PROFILES)
print(f"{len(SAMPLE_ENGINEER_PROFILES)}명 프로필 업로드 완료")

9명 프로필 업로드 완료


In [5]:
# capability만 검색
print("=== capability 검색: 'Java / Spring' ===")
cap_results = store.search_capability("Java / Spring", top_k=5)
for r in cap_results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  | {r.document.page_content[:60].strip()}")

print()

# experience만 검색
print("=== experience 검색: '제조업 ERP 시스템 개발 경험' ===")
exp_results = store.search_experience("제조업 ERP 시스템 개발 경험", top_k=5)
for r in exp_results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  | {r.document.page_content[:60].strip()}")

=== capability 검색: 'Java / Spring' ===
  eng-002   score: 0.9576  | Java / Spring Boot / Spring Security / MySQL / Kafka / AWS
  eng-001   score: 0.9091  | Java / Spring Boot / Spring Batch / PostgreSQL / Redis / Doc
  qa-001    score: 0.8911  | Selenium / Appium / JMeter / Jira / Confluence / TestRail
  eng-005   score: 0.8876  | Python / FastAPI / MongoDB / Docker / Kubernetes / Celery
  eng-004   score: 0.8729  | React / JavaScript / Vue.js / Chart.js / CSS / Webpack

=== experience 검색: '제조업 ERP 시스템 개발 경험' ===
  eng-001   score: 0.8544  | [소개]
MSA 아키텍처 기반 백엔드 전문. 제조업/물류 도메인 경험 다수.

[프로젝트 경험]
현대모비스(
  eng-002   score: 0.8309  | [소개]
자동차/반도체 제조 도메인 백엔드 개발. 이벤트 기반 아키텍처 경험.

[프로젝트 경험]
현대차(2
  eng-005   score: 0.8110  | [소개]
물류/이커머스 도메인 백엔드 개발. MSA 및 컨테이너 오케스트레이션 경험.

[프로젝트 경험]
쿠
  eng-004   score: 0.8100  | [소개]
제조/광고 도메인 프론트엔드 개발. 차트 기반 대시보드 경험.

[프로젝트 경험]
포스코(2024~
  eng-003   score: 0.8017  | [소개]
자동차/금융 도메인 프론트엔드 전문. 대시보드 및 데이터 시각화 경험 풍부.

[프로젝트 경험]
기


In [6]:
# 통합 검색: capability + experience 가중 합산
# PL 포지션 (경험 중심): capability 0.2, experience 0.8
print("=== PL 포지션 검색 (cap=0.2, exp=0.8) ===")
results = store.search(
    capability_query="Java / Spring",
    experience_query="제조업 ERP 시스템 개발. PL 역할. 팀장 역할 필수",
    weights=(0.2, 0.8),
    top_k=5,
    engineer_role="DEVELOPER",
    grades=["SENIOR", "INTERMEDIATE"],
)
for r in results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  grade: {r.metadata['grade']}")

print()

# 백엔드 개발자 포지션 (스킬 중심): capability 0.7, experience 0.3
print("=== 백엔드 개발자 검색 (cap=0.7, exp=0.3) ===")
results = store.search(
    capability_query="Java / Spring",
    experience_query="현대차 ERP 시스템 개발. 백엔드 개발자. 현대차 프로젝트 경험 우대",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
    grades=["SENIOR", "INTERMEDIATE"],
)
for r in results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  grade: {r.metadata['grade']}")

=== PL 포지션 검색 (cap=0.2, exp=0.8) ===
  eng-001   score: 0.8852  grade: SENIOR
  eng-002   score: 0.8751  grade: INTERMEDIATE
  eng-005   score: 0.8488  grade: INTERMEDIATE
  eng-003   score: 0.8399  grade: SENIOR

=== 백엔드 개발자 검색 (cap=0.7, exp=0.3) ===
  eng-002   score: 0.9379  grade: INTERMEDIATE
  eng-001   score: 0.9065  grade: SENIOR
  eng-005   score: 0.8801  grade: INTERMEDIATE
  eng-003   score: 0.8626  grade: SENIOR


In [7]:
# 프론트엔드 개발자 검색 + 프리랜서 제외 (only_full_time=True)
print("=== 프론트엔드 개발자 검색 (정규직만) ===")
results = store.search(
    capability_query="React / Chart.js",
    experience_query="대시보드 개발 경험. 프론트엔드 개발자. 차트.js 경험 우대",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
)
for r in results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  type: {r.metadata['employment_type']}")

print()

# 프리랜서 포함
print("=== 프론트엔드 개발자 검색 (프리랜서 포함) ===")
results = store.search(
    capability_query="React / Chart.js",
    experience_query="대시보드 개발 경험. 프론트엔드 개발자. 차트.js 경험 우대",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
    only_full_time=False,
)
for r in results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  type: {r.metadata['employment_type']}")

=== 프론트엔드 개발자 검색 (정규직만) ===
  eng-003   score: 0.9150  type: FULL_TIME
  eng-002   score: 0.8480  type: FULL_TIME
  eng-005   score: 0.8462  type: FULL_TIME
  eng-001   score: 0.8319  type: FULL_TIME

=== 프론트엔드 개발자 검색 (프리랜서 포함) ===
  eng-004   score: 0.9436  type: FREELANCER
  eng-003   score: 0.9150  type: FULL_TIME
  eng-002   score: 0.8480  type: FULL_TIME
  eng-005   score: 0.8462  type: FULL_TIME
  eng-001   score: 0.8319  type: FULL_TIME


In [8]:
# exclude_ids 테스트: eng-001 제외하고 재검색
print("=== eng-001 제외 후 PL 재검색 ===")
results = store.search(
    capability_query="Java / Spring",
    experience_query="제조업 ERP 시스템 개발. PL 역할.",
    weights=(0.2, 0.8),
    top_k=3,
    engineer_role="DEVELOPER",
    exclude_ids=["eng-001"],
)
for r in results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}")
print("→ eng-001이 결과에서 제외되었는지 확인")

=== eng-001 제외 후 PL 재검색 ===
  eng-002   score: 0.8883
  eng-005   score: 0.8622
  eng-003   score: 0.8496
→ eng-001이 결과에서 제외되었는지 확인
